# Review of how Adam and AdamW Optimizer Algorithms

This series of notebooks will dive into a relatively simple change where we'll write a custom RawKernel that performs weight regularization and perform the calculation that includes weight decay from optimizer **AdamW**. The combination of adding these kernels together into a single pass allows for less trips in vram, overall consituting in higher performance gain. 

Lets first dive into theory:

## Adam

**Adam(Adaptive Moment Estimation)** combines the principles of **Momentum** and **RMSprop** (scaling learning rates inversely by gradient magnitudes).

The full formula for the $t^\text{th}$ update is:
$$
\theta_t = \theta_{t-1} - \frac{\alpha}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t
$$

### 1. The Running Statistics (Moments)
First, we want to understand the running statistics for each weight. At each iteration $t$, we'll compute the gradient $g_t = \nabla_\theta \mathcal{L}(\theta_{t-1})$. Instead of updating the parameters directly with $g_t$, Adam will track two exponentially decaying running averages:

* **First Moment ($m_t$) - Mean / Momentum**:
$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
* This acts like velocity in the real world. In regards to the loss, if the gradient points in the same direction across multiple steps, $m_t$ builds up speed. However, if it fluctuates randomly, the total velocity cancels out as opposing gradients cancel out. We pass in $\beta_1 = 0.9$
    * For the $\beta_1$ hyperparameter, this could be thought of as, how much of past iterations matter to us, a very low $\beta_1$ like `0.0` or `0.1` means that our old weight updates mean practically nothing to us, but `0.9` implies our previous weight updates have high impact and decay slowly. 
    * A good analogy for this momentum is like a *ball* rolling down an $N$ dimensional hill; at steep losses our *ball* will accelerate and gain velocity, but on small loss steps our ball will lose velocity, changes in direction will also slow down our momentum.  

* **Second Moment ($v_t$) - Uncentered Variance**:

$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$

* We pass in a new hyperparameter $\beta_2$ (usually `0.999`), and we'll square the gradient $g^2 _t$ elementwise. In this step, $v_t$ will measure the magnitude/volatility of the **gradients** (our slopes) for each individual parameter. 
    * If a weight experiences mild, steady gradients (e.g., $g_t = 0.01$), the squared gradient is tiny ($0.01^2 = 0.0001$), resulting in a small $v_t$.
    * If a weight experiences volatile, massive gradients (e.g., $g_t = 50.0$), the squared gradient is huge ($50^2 = 2500$), resulting in a large $v_t$.

### 2. Bias Correction 

In the original paper, they address the issue of $m_0$ and $v_0$ having values initialized to zeros. Because the running vectors are initialized to zeros ($m_0 = \mathbf{0}, v_0 = \mathbf{0}$), both statistics are heavily biased toward zero during the earliest optimization steps. For instance, on step $t=1$ with $\beta_2 = 0.999$:

$$v_1 = 0.999(0) + (1 - 0.999)g_1^2 = 0.001 g_1^2$$

without correction, $v1$ would be three orders of magnitude smaller than the actual gradient variance, causing updated gradients to explode. These uncorrected moments distort the step size because both our velocity and variance in the denominator are artificially supressed 

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

With this version:
* At $t = 1$: $\hat{v}_1 = \frac{0.001 g_1^2}{1 - 0.999^1} = \frac{0.001 g_1^2}{0.001} = g_1^2$, where we rescale the statistic to match the initial observation. 
* As the number of training steps increases,  $t \to \infty$: $\beta^t \to 0$, the denominator tends to 1, as a result the bias correction fades out throughout training. 

We now arrive at a new version of our optimizer update step. Normally, in normal SGD this would look like 

$$
\theta_t = \theta_{t-1} - \eta g_t
$$
Then introducing momentum would look something like
$$v_t = \gamma v_{t-1} + \eta g_t$$
$$\theta_t = \theta_{t-1} - v_t$$

Finally, taking the template of these two we arrive at our known optimizer step update

$$\theta_t = \theta_{t-1} - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

* $\eta$: The learning rate (hyperparameter)
* $\epsilon$: A small term (`1e-7`)that's hardcoded in the step update to avoid division by zero, in the framework this is something the user has access to to change, but in reality its best to leave the clip alone, as it doesn't impact our performance. 
* Coordinate-wise adaptation: If a parameter sees huge gradients, $\sqrt{\hat{v}_t}$ is large, which throttles down the effective step size to prevent explosions. Conversely, if a parameter receives sparse or tiny gradients, $\sqrt{\hat{v}_t}$ is small, boosting the step size.

### But How does AdamW Work? 

**AdamW** was created because **L2** regularization pollutes the $m_t$ and $v_t$ moments. Regularization happens before accessing these values, meaning in terms of the equations, providing the penalty makes the gradient become

$$
g_t = \nabla_\theta \mathcal{L}(\theta_{t-1}) + \lambda \theta_{t-1}
$$

The term $\lambda \theta_{t-1}$ will get squared inside $v_t$, where in the final step update, because we have the denominator ($\sqrt{\hat v _t} + \epsilon$) the magnitude of the denominator becomes large, which suppresses and shrinks the weight decay effect for the parameters who literally need harsher updates. Conversly, the parameters with tiny gradients end up receiving proportionally higher weight decay penalties than intended. 

To combat this issue, some genius people decided to update the final loss step with a new hyperparameter `weight_decay`, where we treat the weight shrinkage as an independent subtraction, meaning our two moments stay pure, and we're still able to punish weights.

That means, we'll add an extra term into the mix, which is scaled by the weight decay hyperparameter $\lambda$, the learning rate $\eta$, and use these scalars against our previous weight update $\theta_{t-1}$

$$\theta_t = \theta_{t-1} - \eta \lambda \theta_{t-1} - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

Inside the actual kernel launch that we'll implement later, its best we minimize the amount of `ADD` and `MUL` operations we do, so we can factor out the current weight $\theta_{t-1}$ to make:
$$\theta_t = (1 - \eta \lambda) \theta_{t-1} - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

This will become our final equation that we will end up implementing.

We want to end up writing a kernel that handles all these things for the weights and then the biases. To do that, lets first look at how our normal vectorized implementation works:

In [ ]:
import aether.config as config
import numpy as np
class Adam():
    def __init__(self, learning_rate=.001, decay=0., epsilon=1e-7, beta_1=0.9, beta_2=.999):

        super().__init__(learning_rate, decay)
        self.epsilon = epsilon
        self.beta_1 = beta_1
        self.beta_2 = beta_2  # used to be known as our rho

        self._step_impl = self._step_fallback
        # Bound in _compile_for_device: the memoized fused RawKernel and its
        # vendor variant ('cuda'/'hip'). None on CPU/NumPy or if compilation failed.
        self._adamw_kernel = None
        self._variant = None

    def step(self):
        """Unified optimizer entry point executed once per training step"""
        if self.decay:
            self.current_learning_rate = np.float32(
                self.learning_rate * (1.0 / (1.0 + self.decay * self.iterations))
            )
        t = self.iterations + 1
        bias_correction_1 = np.float32(1.0 - (self.beta_1 ** t))
        bias_correction_2 = np.float32(1.0 - (self.beta_2 ** t))

        self._step_impl(bias_correction_1, bias_correction_2)

        self.iterations += 1

    @staticmethod
    def _l1_subgradient(param, l1_lambda, xp):
        return l1_lambda * xp.where(param < 0, -1.0, 1.0).astype(param.dtype)

    def _get_regularized_gradients(self, layer, xp):
        dweights = layer.dweights
        dbiases = layer.dbiases

        if layer.weight_regularizer_l1 > 0:
            dweights = dweights + self._l1_subgradient(layer.weights, layer.weight_regularizer_l1, xp)
        if layer.weight_regularizer_l2 > 0:
            dweights = dweights + layer.weight_regularizer_l2 * layer.weights

        if layer.bias_regularizer_l1 > 0:
            dbiases = dbiases + self._l1_subgradient(layer.biases, layer.bias_regularizer_l1, xp)
        if layer.bias_regularizer_l2 > 0:
            dbiases = dbiases + layer.bias_regularizer_l2 * layer.biases

        return dweights, dbiases

    def _resolve_weight_decay(self, layer):
        if getattr(layer, "no_weight_decay", False):
            return 0.0
        return getattr(self, "weight_decay", 0.0)

    def _step_fallback(self, bias_correction_1, bias_correction_2):
        xp = config.xp
        learning_rate = np.float32(self.current_learning_rate)
        epsilon = np.float32(self.epsilon)
        beta_1 = np.float32(self.beta_1)
        beta_2 = np.float32(self.beta_2)
        one_minus_beta_1 = np.float32(1.0) - beta_1
        one_minus_beta_2 = np.float32(1.0) - beta_2

        for layer in self.layers:
            dweights, dbiases = self._get_regularized_gradients(layer, xp)

            # Decoupled weight decay (AdamW)
            weight_decay = np.float32(self._resolve_weight_decay(layer))
            if weight_decay > 0.0:
                layer.weights -= learning_rate * weight_decay * layer.weights

            # Update weight momentums and second moment cache
            layer.weight_momentums = beta_1 * layer.weight_momentums + one_minus_beta_1 * dweights
            layer.weight_cache = beta_2 * layer.weight_cache + one_minus_beta_2 * (dweights ** 2)

            weight_momentums_corrected = layer.weight_momentums / bias_correction_1
            weight_cache_corrected = layer.weight_cache / bias_correction_2

            layer.weights -= learning_rate * weight_momentums_corrected / (
                xp.sqrt(weight_cache_corrected) + epsilon
            )

            # Update bias momentums and cache (if biases are present)
            if dbiases is not None:
                layer.bias_momentums = beta_1 * layer.bias_momentums + one_minus_beta_1 * dbiases
                layer.bias_cache = beta_2 * layer.bias_cache + one_minus_beta_2 * (dbiases ** 2)

                bias_momentums_corrected = layer.bias_momentums / bias_correction_1
                bias_cache_corrected = layer.bias_cache / bias_correction_2

                layer.biases -= learning_rate * bias_momentums_corrected / (
                    xp.sqrt(bias_cache_corrected) + epsilon
                )

            # Invalidate any low-precision compute casts stored on the layer
            if hasattr(layer, "invalidate_shadow_caches"):
                layer.invalidate_shadow_caches()

### Implementation for the step algorithm

To avoid breaking the **DRY** principle, we'll pass in the hyperparameter `weight_decay` from **AdamW** straight into both the gpu and fallback paths of the step operation, that way we don't have to write two extra paths over the two we already have. After the Model class holds a list of the trainable layers, we'll pass this into the optimizer after the end of the backward pass of the architecture, we'll have this general algorithm:

1. **Unified Helper Method Step**: 
    * We'll resolve the weight decay scaler combined with the learning rate seen in the formula, $\text{learning\_rate}=\lambda \times \eta$. This is then passed into the implementation. Next, we'll increment the current iteration by 1, and then precompute our denominator for the two bias correction equations. As a reminder:
    $$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

    * `bias_correction_1` = $1 - \beta_1^t$
    * `bias_correction_2` = $1 - \beta_2^t$

    Now we can call our implementation.

2. **_step_impl**:

    In this section the general algorithm becomes resolving the weight regularization passed in from the respective trainable layers hyperparameters **L1** or **L2**. Then, while we're at it we'll precompute the weight decay term in the final parameter update, and finally find $v_t$, $m_t$, compute their corrections, and then install the final weight update. Since everything here is elementwise, the actual implementation is quite straightforward

#### Resolving L1 and L2 Regularization

In normal Adam updates, we've already shown that the weight update happens before we do any calcuations with the two moments:
$$
g_t = \nabla_\theta \mathcal{L}(\theta_{t-1}) + \lambda \theta_{t-1}
$$
In this scenario, the backward pass of the L2 penalty becomes a linear term, which we'll account for in the line below:

```python
if layer.weight_regularizer_l1 > 0:
    dweights = dweights + self._l1_subgradient(layer.weights, layer.weight_regularizer_l1, xp)
if layer.weight_regularizer_l2 > 0:
    dweights = dweights + layer.weight_regularizer_l2 * layer.weights

if layer.bias_regularizer_l1 > 0:
    dbiases = dbiases + self._l1_subgradient(layer.biases, layer.bias_regularizer_l1, xp)
if layer.bias_regularizer_l2 > 0:
    dbiases = dbiases + layer.bias_regularizer_l2 * layer.biases
```
The if statements include a function call to calculating the l1 regularization. For L1, the total gradient after applying regularization becomes: 
$$g_t = \nabla_\theta \mathcal{L}_{\text{data}}(\theta_{t-1}) + \lambda \operatorname{sign}(\theta_{t-1})$$

This is quite simple to explain in english, if the previous weight update is a negative value, then the penalty slope is a constant $-\lambda$, otherwise the previous weight is positive thus we apply a constant $\lambda$. The function call is below 
```python
def _l1_subgradient(self, param, l1_lambda, xp):
    return l1_lambda * xp.where(param < 0, -1.0, 1.0).astype(param.dtype)
```
We'll hold our lambda value, apply a where function that iterates through the tensor elementwise and applies the condition, finally we'll force `float32` to prevent accidental upcasting.

### Resolving the two moment updates

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$

This part is again pretty simple, we can directly write the line as such

```python
one_minus_beta_1 = 1 - beta_1
one_minus_beta_2 = 1 - beta_2

for layer in self.layers:
    # moment one
    layer.weight_momentums = beta_1 * layer.weight_momentums + one_minus_beta_1 * dweights
    
    # moment two
    layer.weight_cache = beta_2 * layer.weight_cache + one_minus_beta_2 * (dweights ** 2)
```

Now that we've computed $m_t$ and $v_t$, we need to compoute $\hat m_t$ and $\hat v_t$. For us, we've already computed the denominator of this equation, meaning the next line becomes

```python
    layer.weight_momentums_corrected = layer.weight_momentums / bias_correction_1
    layer.weight_cache = layer.weight_cache / bias_correction_2
```

We've already computed all the parts we need to actually perform the final update. We've collected $\hat m_t$, $\hat v_t$, saved $\lambda \eta$ and $\eta$, and our clip value $\epsilon$. Thus, our AdamW update (which works for Adam as well):

$$\theta_t = (1 - \eta \lambda) \theta_{t-1} - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

We can write the final step below

```python
    layer.weights -= learning_rate * weight_momentums_corrected / xp.sqrt(weight_cache_corrected) + epsilon
```

This concludes the review over the vectorized implementation of the **Adam** and **AdamW** passes. This gives us valuable insight as to how we will approach a custom RawKernel to reduce the number of passes in total. 
